In [ ]:
# this notebook is for assembling model inputs for conversation modeling (GPT embeddings as well as spike counts/frs)

In [1]:
# module imports
from glob import glob
import pandas as pd
import os
import numpy as np
from neo.io import BlackrockIO
import soundfile as sf

In [2]:
# set dirs
data_dir = "/mnt/labworlds/Hayden/Hayden_Lab/speech_247/podcast_comparison"

In [39]:
# get patient dirs
patient_dirs = glob(f"{data_dir}/Y*")
patient_dirs = {os.path.basename(dir_):dir_ for dir_ in patient_dirs}

In [40]:
# load blackrock ios
patient_ios = {}
for pt, dir_ in patient_dirs.items():
    io_path = glob(f"{dir_}/nsp/*/*NSP-2.ns5")[0]
    io = BlackrockIO(io_path)
    patient_ios[pt] = io

In [28]:
# get audio signal and output to wave file
patient_audios = {}
for pt, io in patient_ios.items():
    dir_ = patient_dirs[pt]
    save_path = f'{dir_}/whisperx_out/podcast_audio.wav'
    if os.path.exists(save_path):
        print("already processed", pt)
        #continue
    print("starting pt", pt)
    block = io.read_block()
    signal = block.segments[0].analogsignals[1]
    channel_names = signal.name[11:-1].split()
    room_mic_id = channel_names.index('Audio')
    if pt == "YFF":
        signal = np.vstack([signal, block.segments[1].analogsignals[1]])
    if room_mic_id == -1:
        print("AUDIO NOT FOUND FOR", pt)
    audio = np.array(signal[:, room_mic_id])
    audio = audio / np.max(np.abs(audio))
    
    sf.write(save_path, audio, 30_000)

already processed YEZ
starting pt YEZ
already processed YEY
starting pt YEY
